# L41 · 加窗全流程 — 完整 FFT 管线

**目标**：理解信号截断导致的频谱泄漏，掌握 Hann 窗的压制原理，完成 `windowed_fft` 并用它观察真实旁瓣差异。

🔗 **Aurora 连接**：`aurora.audio.stft()` 在每帧调用加窗 FFT；`aurora/audio/windows.py` 提供 `hann`、`hamming` 等窗函数，正是本节要接入的模块。

FFT 假设输入信号在帧边界处周期延拓；如果截取的帧恰好不是整数周期，边界处就产生跳变，相当于在频域与 sinc 函数做卷积，能量从主瓣"泄漏"到全部频率 bin。加窗就是在时域给帧乘一个两端平滑为零的权重序列，把跳变抹掉，代价是主瓣略微变宽。Hann 窗是工程中最常用的折中方案：旁瓣比矩形窗低约 30 dB，主瓣只宽一倍。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from aurora.audio.windows import hann, hamming

## 1. 频谱泄漏

截断信号 = 原始信号 × 矩形窗。时域相乘 ↔ 频域卷积，所以频谱 = 原始频谱 ∗ sinc。sinc 的旁瓣从主瓣向两侧延伸，把本应集中在一个 bin 的能量分散到所有 bin，这就是**频谱泄漏（spectral leakage）**。

矩形窗的旁瓣峰值比主瓣仅低约 `-13 dB`，意味着一个强信号的旁瓣足以淹没旁边的弱信号。

In [ ]:
# 演示：一个恰好不在整数 bin 的正弦波的频谱泄漏
N = 256
fs = 1000.0
# 440 Hz 不是 fs/N 的整数倍 → 频率不对齐 bin
t = np.arange(N) / fs
x = np.sin(2 * np.pi * 440 * t)

X_rect = np.abs(np.fft.rfft(x))
freqs = np.fft.rfftfreq(N, 1/fs)

plt.figure(figsize=(10, 3))
plt.plot(freqs, 20 * np.log10(X_rect + 1e-12))
plt.axvline(440, color='r', linestyle='--', label='440 Hz')
plt.xlabel('频率 (Hz)')
plt.ylabel('幅度 (dB)')
plt.title('矩形窗 FFT — 明显的旁瓣泄漏')
plt.legend()
plt.tight_layout()
plt.show()

## 2. Hann 窗

Hann 窗定义为：

```
w[n] = 0.5 * (1 - cos(2*pi*n / (N-1))),   n = 0, 1, ..., N-1
```

两端值为 0，中间平滑隆起。它的频域响应：
- 主瓣宽度是矩形窗的 **2 倍**（分辨率略降）
- 旁瓣峰值比主瓣低约 **-31 dB**（矩形窗仅 -13 dB），大幅压制泄漏

`aurora/audio/windows.py` 中的 `hann(N)` 返回长度 N 的 Hann 窗向量。

In [ ]:
# 对比三种窗函数的时域形状
N = 64
n = np.arange(N)
w_rect = np.ones(N)
w_hann = hann(N)
w_hamm = hamming(N)

plt.figure(figsize=(10, 3))
plt.plot(n, w_rect, label='矩形窗')
plt.plot(n, w_hann, label='Hann 窗')
plt.plot(n, w_hamm, label='Hamming 窗', linestyle='--')
plt.xlabel('样本 n')
plt.ylabel('权重')
plt.title('三种窗函数时域形状（N=64）')
plt.legend()
plt.tight_layout()
plt.show()

## 3. 加窗 FFT 流程

标准流程分三步：

```
1. frame     = x_segment          # 从信号中取一帧（长度 N）
2. x_win     = frame * window      # 逐点乘窗函数（同长）
3. spectrum  = FFT(x_win)          # 对加窗帧做 FFT，取模得幅度谱
```

在 STFT 中，对每一帧重复这三步，帧与帧之间有 hop_length 的偏移量。`aurora.audio.stft()` 内部就是这个循环。

In [ ]:
# 展示加窗后信号边界平滑
N = 128
t = np.arange(N) / 1000.0
x_seg = np.sin(2 * np.pi * 440 * t)   # 不对齐的帧
w = hann(N)
x_win = x_seg * w

fig, axes = plt.subplots(1, 2, figsize=(12, 3))
axes[0].plot(x_seg, label='原始帧')
axes[0].set_title('截断帧（边界有跳变）')
axes[0].set_xlabel('样本')
axes[1].plot(x_win, label='加 Hann 窗后', color='orange')
axes[1].set_title('Hann 加窗后（边界平滑）')
axes[1].set_xlabel('样本')
plt.tight_layout()
plt.show()

## 4. ✏️ 实现 `windowed_fft(x, window_type="hann")`

**推理路线**：
1. 根据 `window_type` 选择窗函数：`"hann"` → `hann(N)`；`"hamming"` → `hamming(N)`；`"rectangular"` → `np.ones(N)`。
2. `x_win = x * window`，逐点相乘（形状不变）。
3. 返回 `np.fft.fft(x_win)`（复数数组，长度等于 `len(x)`）。

**参考输入输出**：
```python
windowed_fft(np.ones(64), "rectangular")
# 矩形窗不改变 x，结果等价于 np.fft.fft(np.ones(64))
# 预期：bin 0 = 64+0j，其余 bin ≈ 0+0j
```

<details><summary>点击查看参考实现</summary>

```python
def windowed_fft(x: np.ndarray, window_type: str = "hann") -> np.ndarray:
    N = len(x)
    if window_type == "hann":
        window = hann(N)
    elif window_type == "hamming":
        window = hamming(N)
    elif window_type == "rectangular":
        window = np.ones(N)
    else:
        raise ValueError(f"未知窗类型：{window_type}")
    x_win = x * window
    return np.fft.fft(x_win)
```

</details>

In [ ]:
def windowed_fft(x: np.ndarray, window_type: str = "hann") -> np.ndarray:
    """对输入帧 x 应用指定窗函数后执行 FFT。

    参数
    ----
    x           : 输入帧，形状 (N,)
    window_type : 窗类型，"hann" | "hamming" | "rectangular"

    返回
    ----
    复数 FFT 输出，形状 (N,)
    """
    N = len(x)
    # ✏️ TODO: 根据 window_type 生成窗向量（hann / hamming / ones）
    window = ...

    # ✏️ TODO: 加窗
    x_win = ...

    # ✏️ TODO: 返回 FFT 结果
    return ...

In [ ]:
# 检查：矩形窗不改变 x，结果与直接 FFT 完全一致
assert np.allclose(
    windowed_fft(np.ones(64), "rectangular"),
    np.fft.fft(np.ones(64))
), "矩形窗 FFT 结果应与 np.fft.fft 完全相等"
print('✅ windowed_fft(rectangular) 通过')

# 检查：hann 窗输出长度与输入一致
x_test = np.random.randn(128)
out = windowed_fft(x_test, 'hann')
assert out.shape == (128,), f"输出形状错误：{out.shape}"
print('✅ windowed_fft(hann) 输出形状正确')

# 检查：bin 0 实部等于 sum(x_win)
w = hann(128)
assert np.isclose(out[0].real, np.sum(x_test * w)), "bin 0 应等于加窗信号之和"
print('✅ bin 0 = sum(x * window) 验证通过')

## 5. 参数实验：旁瓣对比

**实验设置**：440 Hz 正弦波，采样率 1000 Hz，帧长 256 点。440 Hz 不是 bin 频率分辨率（1000/256 ≈ 3.9 Hz）的整数倍，频率不对齐 bin，泄漏最严重。

**预期现象**：
- `"rectangular"`：主瓣两侧旁瓣高，第一旁瓣比主瓣仅低约 13 dB
- `"hann"`：旁瓣压制明显，第一旁瓣比主瓣低约 31 dB，两条曲线在主瓣附近差距肉眼可见

调整 `freq_hz` 到恰好对齐某个 bin（如 500 Hz = 128 个整周期）后，两种窗的频谱都会退化成单根线，泄漏消失。

In [ ]:
# 参数实验：对比 rectangular vs hann 旁瓣
freq_hz = 440.0   # 修改此值观察对齐/非对齐的差异
fs = 1000.0
N = 256

t = np.arange(N) / fs
x = np.sin(2 * np.pi * freq_hz * t)

X_rect = np.abs(windowed_fft(x, window_type="rectangular")[:N//2])
X_hann = np.abs(windowed_fft(x, window_type="hann")[:N//2])
freqs = np.fft.rfftfreq(N, 1/fs)[:-1]   # N//2 点

# 归一化：主瓣峰值设为 0 dB
X_rect_db = 20 * np.log10(X_rect / X_rect.max() + 1e-12)
X_hann_db = 20 * np.log10(X_hann / X_hann.max() + 1e-12)

# 计算旁瓣差（排除主瓣附近 ±5 bin）
peak_bin = np.argmax(X_rect)
mask = np.ones(len(X_rect), dtype=bool)
mask[max(0, peak_bin-5):peak_bin+6] = False
sidelobe_rect = X_rect_db[mask].max()
sidelobe_hann = X_hann_db[mask].max()
print(f'矩形窗最大旁瓣: {sidelobe_rect:.1f} dB')
print(f'Hann 窗最大旁瓣:  {sidelobe_hann:.1f} dB')
print(f'旁瓣压制差: {sidelobe_rect - sidelobe_hann:.1f} dB')

plt.figure(figsize=(11, 4))
plt.plot(freqs, X_rect_db, label='矩形窗', alpha=0.8)
plt.plot(freqs, X_hann_db, label='Hann 窗', alpha=0.8)
plt.axhline(-13, color='gray', linestyle=':', linewidth=0.8, label='-13 dB 参考')
plt.axhline(-31, color='brown', linestyle=':', linewidth=0.8, label='-31 dB 参考')
plt.ylim(-80, 5)
plt.xlabel('频率 (Hz)')
plt.ylabel('归一化幅度 (dB)')
plt.title(f'频谱泄漏对比（{freq_hz} Hz，fs={fs} Hz，N={N}）')
plt.legend()
plt.tight_layout()
plt.show()

## 收束

本节实现了 `windowed_fft(x, window_type)`，它在加窗后输出长度为 N 的复数 FFT 频谱。Hann 窗将旁瓣压低约 30 dB 的能力来自 `aurora/audio/windows.py` 中 `hann(N)` 的余弦权重设计。下一节（Week 3 / STFT）将把 `windowed_fft` 嵌入帧循环，配合 `hop_length` 生成时频矩阵，供 mel 滤波器组和 MFCC 使用。